In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Installations ##

In [ ]:
!git remote add origin https://github.com/nakwei/finBERT-2.0.git
!git branch -M main
!git push -u origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
from __future__ import absolute_import, division, print_function

!pip install datasets --upgrade
from datasets.load import load_dataset
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from nltk.tokenize import sent_tokenize
from transformers import (
    TFBertForSequenceClassification,
    BertTokenizerFast,
    create_optimizer
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import KFold

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

## Slanted LR

In [ ]:
def compute_layer_multiplier(var_name, d_rate=2.0):
    # Extract layer index from var name e.g. 'encoder/layer_._5/' -> 5
    import re
    m = re.search(r'encoder/layer\._(\d+)', var_name)
    if m:
        layer = int(m.group(1))
        # layers nearer to top (11) get lr * d_rate^(11-layer)
        return d_rate ** (11 - layer)
    # embeddings get extra small LR
    if 'embeddings' in var_name:
        return d_rate ** 12
    return 1.0

class SlantedTriangularSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, max_lr, total_steps, cut_frac=0.1, ratio=32):
        self.max_lr = max_lr
        self.total_steps = tf.cast(total_steps, tf.float32)
        self.cut = cut_frac * self.total_steps
        self.ratio = ratio
        self.min_lr = max_lr / ratio

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        def lr_inc():  # warmup
            return self.min_lr + (self.max_lr - self.min_lr) * (step / self.cut)
        def lr_dec():  # decay
            return self.max_lr - (self.max_lr - self.min_lr) * ((step - self.cut) / (self.total_steps - self.cut))
        return tf.where(step < self.cut, lr_inc(), lr_dec())

## Config ##

In [ ]:
class Config(object):
    def __init__(
        self,
        data_dir=None,
        bert_model='bert-base-uncased',
        model_dir='model_out',
        hf_dataset_name=None,
        hf_config_name=None,
        max_seq_length=64,
        train_batch_size=32,
        eval_batch_size=32,
        learning_rate=5e-5,
        num_train_epochs=12,
        warmup_proportion=0.1,
        seed=42,
        output_mode='classification',
        base_model='bert-base-uncased'
    ):
        self.data_dir = data_dir
        self.bert_model = bert_model
        self.model_dir = model_dir
        self.hf_dataset_name = hf_dataset_name
        self.hf_config_name = hf_config_name
        self.max_seq_length = max_seq_length
        self.train_batch_size = train_batch_size
        self.eval_batch_size = eval_batch_size
        self.learning_rate = learning_rate
        self.num_train_epochs = num_train_epochs  # match number of encoder layers
        self.warmup_proportion = warmup_proportion
        self.seed = seed
        self.output_mode = output_mode
        self.base_model = base_model
        random.seed(self.seed)
        np.random.seed(self.seed)
        tf.random.set_seed(self.seed)



## Gradual Unfreezing ##

In [ ]:
class GradualUnfreezeCallback(tf.keras.callbacks.Callback):
    def __init__(self, total_layers, epochs):
        super().__init__()
        self.total_layers = total_layers
        self.total_third_epochs = epochs * 3
        self.layers_unfrozen = 0
        self.steps_per_epoch = None
        self.batches_per_third_epoch = None

    def on_train_begin(self, logs=None):
        self.steps_per_epoch = self.params['steps']
        self.batches_per_third_epoch = self.steps_per_epoch // 3
        self.total_batches = self.batches_per_third_epoch * self.total_third_epochs
        self.next_unfreeze_batch = self.batches_per_third_epoch
        self.current_batch = 0
        print(f"Unfreezing will occur every {self.batches_per_third_epoch} batches.")

    def on_batch_end(self, batch, logs=None):
        self.current_batch += 1

        if (self.current_batch >= self.next_unfreeze_batch and
            self.layers_unfrozen < self.total_layers):

            layer_to_unfreeze = self.layers_unfrozen
            self.model.bert.encoder.layer[layer_to_unfreeze].trainable = True
            print(f"Unfroze {layer_to_unfreeze} out of {self.total_layers}")

            self.layers_unfrozen += 1
            self.next_unfreeze_batch += self.batches_per_third_epoch


def freeze_all_encoder_layers(model):
    # Freeze all BERT encoder layers
    for layer in model.bert.encoder.layer:
        layer.trainable = False

## Finbert Init ##

In [ ]:
class FinBertTF(object):
    def __init__(self, config, label_list):
        self.config = config
        self.label_list = label_list
        self.num_labels = len(label_list)
        self.tokenizer = BertTokenizerFast.from_pretrained(
            config.base_model, do_lower_case=True
        )
        self.model = TFBertForSequenceClassification.from_pretrained(
            config.bert_model,
            num_labels=self.num_labels
        )
        self.optimizer, self.lr_schedule = None, None

    def _encode_dataset(self, texts, labels, shuffle=False, batch_size=32):
        """Encodes a dataset of texts and labels into a TensorFlow Dataset."""
        ds = self._encode_examples(texts, labels)
        if shuffle:
            ds = ds.shuffle(10000)  # Adjust buffer size as needed
        return ds.batch(batch_size)

    def _encode_examples(self, texts, labels=None):
        encodings = self.tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=self.config.max_seq_length,
            return_tensors='tf'
        )
        if labels is not None:
            ds = tf.data.Dataset.from_tensor_slices((
                dict(encodings),
                tf.convert_to_tensor(labels)
            ))
        else:
            ds = tf.data.Dataset.from_tensor_slices(dict(encodings))
        return ds

    def _get_hf_dataset(self, split):
        # load from HuggingFace
        ds = load_dataset(
            self.config.hf_dataset_name,
            self.config.hf_config_name,
            split=split
        )
        texts = ds['sentence']
        labels = ds['label']
        tf_ds = self._encode_examples(texts, labels)
        batch_size = self.config.train_batch_size if split=='train' else self.config.eval_batch_size
        if split == 'train':
            tf_ds = tf_ds.shuffle(10000)
        return tf_ds.batch(batch_size)

    def _get_csv_dataset(self, phase):
        file_path = os.path.join(self.config.data_dir, f"{phase}.csv")
        df = pd.read_csv(file_path, sep='\t', index_col=False)
        texts = df['text'].astype(str).tolist()
        labels = [self.label_list.index(l) for l in df['label']]
        ds = self._encode_examples(texts, labels)
        if phase == 'train':
            ds = ds.shuffle(10000)
            return ds.batch(self.config.train_batch_size)
        return ds.batch(self.config.eval_batch_size)

    def get_dataset(self, phase):
        if self.config.hf_dataset_name:
            # HF splits: 'train', 'test', 'validation'
            return self._get_hf_dataset(phase)
        else:
            return self._get_csv_dataset(phase)

    def train(self):
        # Load full dataset dict and extract 'train'
        if not self.config.hf_dataset_name:
            raise ValueError("hf_dataset_name must be provided for HF flow.")
        ds_dict = load_dataset(
            self.config.hf_dataset_name,
            self.config.hf_config_name
        )
        raw_train = ds_dict['train']

        # create splits: 80% train_val, 20% test
        split1 = raw_train.train_test_split(test_size=0.2, seed=self.config.seed)
        train_val = split1['train']
        test_raw = split1['test']
        # split train_val into train (80%) and val (20%)
        split2 = train_val.train_test_split(test_size=0.2, seed=self.config.seed)
        train_raw = split2['train']
        val_raw = split2['test']

        # extract texts/labels
        train_texts, train_labels = train_raw['sentence'], train_raw['label']
        val_texts, val_labels = val_raw['sentence'], val_raw['label']
        test_texts, test_labels = test_raw['sentence'], test_raw['label']

        # encode datasets
        train_ds = self._encode_dataset(
            train_texts, train_labels,
            shuffle=True, batch_size=self.config.train_batch_size
        )
        val_ds = self._encode_dataset(
            val_texts, val_labels,
            shuffle=False, batch_size=self.config.eval_batch_size
        )

        # optimizer and compile
        steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()
        total_steps = steps_per_epoch * self.config.num_train_epochs
        warmup_steps = int(total_steps * self.config.warmup_proportion)
        self.optimizer, self.lr_schedule = create_optimizer(
            init_lr=self.config.learning_rate,
            num_train_steps=total_steps,
            num_warmup_steps=warmup_steps,
            weight_decay_rate=0.01
        )

        class_weights = compute_sqrt_inverse_class_weights(train_labels)
        loss_fn = create_weighted_loss(class_weights)

        # loss_fn = (
        #     tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        #     if self.config.output_mode == 'classification'
        #     else tf.keras.losses.MeanSquaredError()
        # )

        metrics = ['accuracy'] if self.config.output_mode == 'classification' else []
        self.model.compile(
            optimizer=self.optimizer,
            loss=loss_fn,
            metrics=metrics
        )

        # train
        self.model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=self.config.num_train_epochs
        )

        # save outputs
        self.model.save_pretrained(self.config.model_dir)
        self.tokenizer.save_pretrained(self.config.model_dir)

        # store test for evaluate
        self._test_data = (test_texts, test_labels)

    def evaluate(self, phase='test'):
        """
        Evaluate model on specified split and compute accuracy, precision, recall, and F1.
        phase: 'validation' or 'test'
        """
        eval_ds = self.get_dataset(phase)
        y_true, y_pred = [], []
        for batch in eval_ds:
            features, labels = batch
            logits = self.model(features, training=False).logits
            preds = tf.argmax(logits, axis=-1).numpy()
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro'
        )
        report = classification_report(
            y_true, y_pred, target_names=self.label_list
        )
        return {
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'report': report
        }

    def predict(self, text, batch_size=8):
        sentences = sent_tokenize(text)
        ds = self._encode_examples(sentences)
        ds = ds.batch(batch_size)
        logits = self.model.predict(ds).logits
        probs = tf.nn.softmax(logits, axis=-1).numpy()
        preds = np.argmax(probs, axis=-1)
        df = pd.DataFrame({
            'sentence': sentences,
            'logit': list(logits),
            'prediction': [self.label_list[p] for p in preds],
            'sentiment_score': probs[:, 0] - probs[:, 1]
        })
        return df

    def cross_validate(self, n_splits=10):
        if not self.config.hf_dataset_name:
            raise ValueError("hf_dataset_name required for cross-validation.")
        ds_dict = load_dataset(
            self.config.hf_dataset_name, self.config.hf_config_name
        )
        raw = ds_dict['train']
        texts = list(raw['sentence'])
        labels = list(raw['label'])

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=self.config.seed)
        accs, precisions, recalls, f1s = [], [], [], []
        total_layers = len(TFBertForSequenceClassification.from_pretrained(
            self.config.bert_model).bert.encoder.layer)


        for fold, (train_idx, val_idx) in enumerate(kf.split(texts), 1):
            print(f"Starting fold {fold}/{n_splits}")
            train_texts = [texts[i] for i in train_idx]
            train_labels = [labels[i] for i in train_idx]
            val_texts = [texts[i] for i in val_idx]
            val_labels = [labels[i] for i in val_idx]

            train_ds = self._encode_dataset(
                train_texts, train_labels,
                shuffle=True, batch_size=self.config.train_batch_size
            )
            val_ds = self._encode_dataset(
                val_texts, val_labels,
                shuffle=False, batch_size=self.config.eval_batch_size
            )

            model = TFBertForSequenceClassification.from_pretrained(
                self.config.bert_model, num_labels=self.num_labels
            )
            freeze_all_encoder_layers(model)

            # weighted loss
            class_weights = compute_sqrt_inverse_class_weights(train_labels)
            loss_fn = create_weighted_loss(class_weights)

            #loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

            steps = tf.data.experimental.cardinality(train_ds).numpy()
            total_steps = steps * self.config.num_train_epochs

            # slanted = SlantedTriangularSchedule(
            #   learning_rate=self.config.learning_rate,
            #   num_train_steps = total_steps,
            #   cut_frac = self.config.warmup_proportion,
            #   ratio = 32
            #   )

            # optimizer = tf.keras.optimizers.Adam(learning_rate=slanted)

            lr_schedule = SlantedTriangularSchedule(
                max_lr=self.config.learning_rate,
                total_steps=total_steps,
                cut_frac=self.config.warmup_proportion,
                ratio=32
            )


            class MultiLRAdam(tf.keras.optimizers.Adam):
                def __init__(self, base_lr, lr_schedule, **kwargs):
                    super().__init__(learning_rate=lr_schedule, **kwargs)
                    self.base_lr = base_lr
                    self.lr_schedule = lr_schedule

                def apply_gradients(self, grads_and_vars, name=None, **kwargs):
                    scaled_grads_and_vars = []
                    for grad, var in grads_and_vars:
                        if grad is not None:
                            lr_multiplier = compute_layer_multiplier(var.name)
                            grad = grad * tf.cast(lr_multiplier, tf.float32)
                        scaled_grads_and_vars.append((grad, var))
                    return super().apply_gradients(scaled_grads_and_vars, name=name, **kwargs)

            optimizer = MultiLRAdam(base_lr=self.config.learning_rate, lr_schedule=lr_schedule)


            model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

            gu = GradualUnfreezeCallback(total_layers, self.config.num_train_epochs)
            model.fit(
                train_ds, validation_data=val_ds,
                epochs=self.config.num_train_epochs,
                callbacks=[gu], verbose=1
            )

            y_true, y_pred = [], []
            for batch in val_ds:
                features, batch_labels = batch
                logits = model(features, training=False).logits
                preds = tf.argmax(logits, axis=-1).numpy()
                y_true.extend(batch_labels.numpy())
                y_pred.extend(preds)

            acc = accuracy_score(y_true, y_pred)
            precision, recall, f1, _ = precision_recall_fscore_support(
                y_true, y_pred, average='macro'
            )
            print(f"Fold {fold} -- Acc: {acc:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}, F1: {f1:.4f}")
            accs.append(acc); precisions.append(precision)
            recalls.append(recall); f1s.append(f1)

        print("\nCross-validation results:")
        print(f"Mean Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
        print(f"Mean Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
        print(f"Mean Recall: {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
        print(f"Mean F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

    def save_model(self, model_path):
        """Saves the model and tokenizer to the specified path."""
        self.model.save_pretrained(model_path)
        self.tokenizer.save_pretrained(model_path)

## Class weighted cross entropy ##


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def compute_sqrt_inverse_class_weights(y_train):
    classes = np.unique(y_train)
    class_freqs = np.bincount(y_train) / len(y_train)
    inv_freqs = 1.0 / class_freqs
    sqrt_inv_freqs = np.sqrt(inv_freqs)
    class_weights = dict(zip(classes, sqrt_inv_freqs))
    return class_weights

def create_weighted_loss(class_weights):
    def weighted_loss(y_true, y_pred):
        weights = tf.gather(tf.constant(list(class_weights.values()), dtype=tf.float32), y_true)
        loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
        weighted_loss = loss * weights
        return tf.reduce_mean(weighted_loss)
    return weighted_loss

## Usage

In [ ]:
config = Config(
        hf_dataset_name='takala/financial_phrasebank',
        hf_config_name='sentences_allagree',
        num_train_epochs=4,
        train_batch_size=16,
        eval_batch_size=16,
    )
label_list = ['positive', 'negative', 'neutral']
fb = FinBertTF(config, label_list)
fb.cross_validate(n_splits=10)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting fold 1/10


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unfreezing will occur every 42 batches.
Epoch 1/4
128/128 [==============================] - 46s 191ms/step - loss: 1.4545 - accuracy: 0.6284 - val_loss: 1.2395 - val_accuracy: 0.7489
Epoch 2/4
128/128 [==============================] - 21s 161ms/step - loss: 1.0008 - accuracy: 0.7472 - val_loss: 0.9751 - val_accuracy: 0.7974
Epoch 3/4
128/128 [==============================] - 21s 164ms/step - loss: 0.8352 - accuracy: 0.7879 - val_loss: 0.8896 - val_accuracy: 0.8150
Epoch 4/4
128/128 [==============================] - 21s 165ms/step - loss: 0.7599 - accuracy: 0.8061 - val_loss: 0.8665 - val_accuracy: 0.8062
Fold 1 -- Acc: 0.8062, Prec: 0.8303, Rec: 0.6157, F1: 0.5954
Starting fold 2/10


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Unfreezing will occur every 42 batches.
Epoch 1/4
128/128 [==============================] - 39s 190ms/step - loss: 1.4690 - accuracy: 0.5667 - val_loss: 1.3263 - val_accuracy: 0.7124
Epoch 2/4
128/128 [==============================] - 21s 167ms/step - loss: 1.0009 - accuracy: 0.7532 - val_loss: 0.9901 - val_accuracy: 0.7699
Epoch 3/4
128/128 [==============================] - 21s 165ms/step - loss: 0.8150 - accuracy: 0.7856 - val_loss: 0.8888 - val_accuracy: 0.7965
Epoch 4/4
128/128 [==============================] - 21s 165ms/step - loss: 0.7406 - accuracy: 0.8091 - val_loss: 0.8442 - val_accuracy: 0.8230
Fold 8 -- Acc: 0.8230, Prec: 0.8449, Rec: 0.6695, F1: 0.6632
Starting fold 9/10


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unfreezing will occur every 42 batches.
Epoch 1/4
128/128 [==============================] - 39s 194ms/step - loss: 1.5065 - accuracy: 0.5682 - val_loss: 1.2201 - val_accuracy: 0.6903
Epoch 2/4
128/128 [==============================] - 21s 164ms/step - loss: 1.0213 - accuracy: 0.7507 - val_loss: 0.9464 - val_accuracy: 0.7876
Epoch 3/4
128/128 [==============================] - 21s 167ms/step - loss: 0.8402 - accuracy: 0.7792 - val_loss: 0.8383 - val_accuracy: 0.8142
Epoch 4/4
128/128 [==============================] - 21s 165ms/step - loss: 0.7577 - accuracy: 0.8057 - val_loss: 0.8070 - val_accuracy: 0.8363
Fold 9 -- Acc: 0.8363, Prec: 0.8216, Rec: 0.6637, F1: 0.6990
Starting fold 10/10


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unfreezing will occur every 42 batches.
Epoch 1/4
128/128 [==============================] - 40s 189ms/step - loss: 1.4181 - accuracy: 0.6340 - val_loss: 1.3251 - val_accuracy: 0.7168
Epoch 2/4
128/128 [==============================] - 21s 165ms/step - loss: 0.9852 - accuracy: 0.7640 - val_loss: 1.0747 - val_accuracy: 0.7434
Epoch 3/4
128/128 [==============================] - 21s 165ms/step - loss: 0.8030 - accuracy: 0.7954 - val_loss: 0.9975 - val_accuracy: 0.7566
Epoch 4/4
128/128 [==============================] - 21s 167ms/step - loss: 0.7240 - accuracy: 0.8238 - val_loss: 0.9555 - val_accuracy: 0.7566
Fold 10 -- Acc: 0.7566, Prec: 0.7072, Rec: 0.5742, F1: 0.5829

Cross-validation results:
Mean Accuracy: 0.7973 ± 0.0278
Mean Precision: 0.7488 ± 0.1030
Mean Recall: 0.6259 ± 0.0480
Mean F1: 0.6251 ± 0.0701


## Results ##

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Starting fold 1/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.6572 - accuracy: 0.5732Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 6s - loss: 1.5563 - accuracy: 0.5934Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4602 - accuracy: 0.6270Unfroze 2 out of 12
128/128 [==============================] - 46s 191ms/step - loss: 1.4545 - accuracy: 0.6284 - val_loss: 1.2395 - val_accuracy: 0.7489
Epoch 2/4
 39/128 [========>.....................] - ETA: 13s - loss: 1.0859 - accuracy: 0.7596Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0257 - accuracy: 0.7562Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0013 - accuracy: 0.7485Unfroze 5 out of 12
128/128 [==============================] - 21s 161ms/step - loss: 1.0008 - accuracy: 0.7472 - val_loss: 0.9751 - val_accuracy: 0.7974
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.7801 - accuracy: 0.8159Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8402 - accuracy: 0.7864Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8452 - accuracy: 0.7836Unfroze 8 out of 12
128/128 [==============================] - 21s 164ms/step - loss: 0.8352 - accuracy: 0.7879 - val_loss: 0.8896 - val_accuracy: 0.8150
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7389 - accuracy: 0.8339Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 7s - loss: 0.7551 - accuracy: 0.8133Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7542 - accuracy: 0.8083Unfroze 11 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.7599 - accuracy: 0.8061 - val_loss: 0.8665 - val_accuracy: 0.8062
Fold 1 -- Acc: 0.8062, Prec: 0.8303, Rec: 0.6157, F1: 0.5954
Starting fold 2/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.6374 - accuracy: 0.5976Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.5182 - accuracy: 0.6092Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4024 - accuracy: 0.6395Unfroze 2 out of 12
128/128 [==============================] - 40s 197ms/step - loss: 1.3962 - accuracy: 0.6416 - val_loss: 1.2701 - val_accuracy: 0.6960
Epoch 2/4
 39/128 [========>.....................] - ETA: 13s - loss: 1.0276 - accuracy: 0.7548Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 0.9875 - accuracy: 0.7608Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 0.9584 - accuracy: 0.7683Unfroze 5 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.9605 - accuracy: 0.7678 - val_loss: 0.9847 - val_accuracy: 0.7753
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.9077 - accuracy: 0.7534Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8193 - accuracy: 0.7856Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.7957 - accuracy: 0.7924Unfroze 8 out of 12
128/128 [==============================] - 21s 168ms/step - loss: 0.7918 - accuracy: 0.7938 - val_loss: 0.9057 - val_accuracy: 0.8062
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.6557 - accuracy: 0.8357Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7073 - accuracy: 0.8287Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7129 - accuracy: 0.8251Unfroze 11 out of 12
128/128 [==============================] - 21s 167ms/step - loss: 0.7187 - accuracy: 0.8208 - val_loss: 0.8676 - val_accuracy: 0.8106
Fold 2 -- Acc: 0.8106, Prec: 0.7577, Rec: 0.7016, F1: 0.7231
Starting fold 3/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 14s - loss: 1.6367 - accuracy: 0.5777Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.5212 - accuracy: 0.6009Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4331 - accuracy: 0.6275Unfroze 2 out of 12
128/128 [==============================] - 39s 190ms/step - loss: 1.4331 - accuracy: 0.6279 - val_loss: 1.2529 - val_accuracy: 0.7401
Epoch 2/4
 39/128 [========>.....................] - ETA: 14s - loss: 1.0855 - accuracy: 0.7596Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0626 - accuracy: 0.7508Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0197 - accuracy: 0.7571Unfroze 5 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 1.0177 - accuracy: 0.7560 - val_loss: 0.9846 - val_accuracy: 0.7841
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.8651 - accuracy: 0.7635Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8797 - accuracy: 0.7611Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8417 - accuracy: 0.7784Unfroze 8 out of 12
128/128 [==============================] - 21s 164ms/step - loss: 0.8372 - accuracy: 0.7776 - val_loss: 0.8905 - val_accuracy: 0.7930
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7726 - accuracy: 0.7875Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7741 - accuracy: 0.7922Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7761 - accuracy: 0.7925Unfroze 11 out of 12
128/128 [==============================] - 21s 166ms/step - loss: 0.7774 - accuracy: 0.7928 - val_loss: 0.8586 - val_accuracy: 0.7930
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Fold 3 -- Acc: 0.7930, Prec: 0.4905, Rec: 0.5832, F1: 0.5313
Starting fold 4/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.6559 - accuracy: 0.6006Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.5545 - accuracy: 0.6054Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4466 - accuracy: 0.6420Unfroze 2 out of 12
128/128 [==============================] - 40s 193ms/step - loss: 1.4408 - accuracy: 0.6436 - val_loss: 1.3125 - val_accuracy: 0.7313
Epoch 2/4
 39/128 [========>.....................] - ETA: 13s - loss: 1.0990 - accuracy: 0.7372Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0560 - accuracy: 0.7461Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0157 - accuracy: 0.7515Unfroze 5 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 1.0157 - accuracy: 0.7511 - val_loss: 1.0396 - val_accuracy: 0.7401
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.8922 - accuracy: 0.7686Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8550 - accuracy: 0.7801Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8385 - accuracy: 0.7794Unfroze 8 out of 12
128/128 [==============================] - 21s 166ms/step - loss: 0.8419 - accuracy: 0.7781 - val_loss: 0.9396 - val_accuracy: 0.7533
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7948 - accuracy: 0.7893Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7479 - accuracy: 0.8076Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7629 - accuracy: 0.8067Unfroze 11 out of 12
128/128 [==============================] - 21s 167ms/step - loss: 0.7602 - accuracy: 0.8071 - val_loss: 0.9013 - val_accuracy: 0.7753
Fold 4 -- Acc: 0.7753, Prec: 0.6871, Rec: 0.6056, F1: 0.6089
Starting fold 5/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.5629 - accuracy: 0.6113Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.4742 - accuracy: 0.6205Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4054 - accuracy: 0.6445Unfroze 2 out of 12
128/128 [==============================] - 39s 190ms/step - loss: 1.3972 - accuracy: 0.6472 - val_loss: 1.1913 - val_accuracy: 0.7566
Epoch 2/4
 39/128 [========>.....................] - ETA: 14s - loss: 1.0610 - accuracy: 0.7372Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0409 - accuracy: 0.7392Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0077 - accuracy: 0.7515Unfroze 5 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 1.0008 - accuracy: 0.7537 - val_loss: 0.9177 - val_accuracy: 0.7788
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.8219 - accuracy: 0.7889Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8542 - accuracy: 0.7729Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8455 - accuracy: 0.7769Unfroze 8 out of 12
128/128 [==============================] - 21s 164ms/step - loss: 0.8491 - accuracy: 0.7743 - val_loss: 0.8105 - val_accuracy: 0.7965
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7887 - accuracy: 0.8054Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7700 - accuracy: 0.8109Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7751 - accuracy: 0.8036Unfroze 11 out of 12
128/128 [==============================] - 21s 166ms/step - loss: 0.7665 - accuracy: 0.8057 - val_loss: 0.7811 - val_accuracy: 0.8009
Fold 5 -- Acc: 0.8009, Prec: 0.7068, Rec: 0.5932, F1: 0.5886
Starting fold 6/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.7850 - accuracy: 0.4482Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.5533 - accuracy: 0.5557Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4559 - accuracy: 0.5945Unfroze 2 out of 12
128/128 [==============================] - 40s 189ms/step - loss: 1.4501 - accuracy: 0.5972 - val_loss: 1.2763 - val_accuracy: 0.7301
Epoch 2/4
 39/128 [========>.....................] - ETA: 14s - loss: 1.0439 - accuracy: 0.7644Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0491 - accuracy: 0.7554Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0031 - accuracy: 0.7576Unfroze 5 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.9977 - accuracy: 0.7571 - val_loss: 1.0223 - val_accuracy: 0.7611
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.8527 - accuracy: 0.7669Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8088 - accuracy: 0.7872Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8083 - accuracy: 0.7836Unfroze 8 out of 12
128/128 [==============================] - 21s 164ms/step - loss: 0.8030 - accuracy: 0.7841 - val_loss: 0.9324 - val_accuracy: 0.8186
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.6912 - accuracy: 0.8250Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7433 - accuracy: 0.8125Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7251 - accuracy: 0.8151Unfroze 11 out of 12
128/128 [==============================] - 21s 166ms/step - loss: 0.7253 - accuracy: 0.8175 - val_loss: 0.9023 - val_accuracy: 0.8230
Fold 6 -- Acc: 0.8230, Prec: 0.8346, Rec: 0.6892, F1: 0.7277
Starting fold 7/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.7806 - accuracy: 0.3857Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.5939 - accuracy: 0.5136Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4851 - accuracy: 0.5730Unfroze 2 out of 12
128/128 [==============================] - 39s 189ms/step - loss: 1.4837 - accuracy: 0.5741 - val_loss: 1.3885 - val_accuracy: 0.7212
Epoch 2/4
 39/128 [========>.....................] - ETA: 13s - loss: 1.1689 - accuracy: 0.7324Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0804 - accuracy: 0.7500Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0275 - accuracy: 0.7566Unfroze 5 out of 12
128/128 [==============================] - 21s 166ms/step - loss: 1.0214 - accuracy: 0.7571 - val_loss: 1.1143 - val_accuracy: 0.7168
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.7970 - accuracy: 0.8057Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8284 - accuracy: 0.7824Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8309 - accuracy: 0.7810Unfroze 8 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.8438 - accuracy: 0.7777 - val_loss: 1.0090 - val_accuracy: 0.7434
Epoch 4/4
 35/128 [=======>......................] - ETA: 15s - loss: 0.7836 - accuracy: 0.8000Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7812 - accuracy: 0.7955Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7778 - accuracy: 0.7973Unfroze 11 out of 12
128/128 [==============================] - 21s 167ms/step - loss: 0.7710 - accuracy: 0.7974 - val_loss: 0.9820 - val_accuracy: 0.7478
Fold 7 -- Acc: 0.7478, Prec: 0.8075, Rec: 0.5632, F1: 0.5304
Starting fold 8/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.7632 - accuracy: 0.4009Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.6011 - accuracy: 0.5060Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4752 - accuracy: 0.5630Unfroze 2 out of 12
128/128 [==============================] - 39s 190ms/step - loss: 1.4690 - accuracy: 0.5667 - val_loss: 1.3263 - val_accuracy: 0.7124
Epoch 2/4
 39/128 [========>.....................] - ETA: 14s - loss: 1.0773 - accuracy: 0.7404Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0417 - accuracy: 0.7415Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 0.9962 - accuracy: 0.7530Unfroze 5 out of 12
128/128 [==============================] - 21s 167ms/step - loss: 1.0009 - accuracy: 0.7532 - val_loss: 0.9901 - val_accuracy: 0.7699
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.8743 - accuracy: 0.7669Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8390 - accuracy: 0.7769Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8127 - accuracy: 0.7867Unfroze 8 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.8150 - accuracy: 0.7856 - val_loss: 0.8888 - val_accuracy: 0.7965
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7084 - accuracy: 0.8250Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7236 - accuracy: 0.8141Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7426 - accuracy: 0.8062Unfroze 11 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.7406 - accuracy: 0.8091 - val_loss: 0.8442 - val_accuracy: 0.8230
Fold 8 -- Acc: 0.8230, Prec: 0.8449, Rec: 0.6695, F1: 0.6632
Starting fold 9/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 13s - loss: 1.7752 - accuracy: 0.4131Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.6275 - accuracy: 0.5105Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.5089 - accuracy: 0.5675Unfroze 2 out of 12
128/128 [==============================] - 39s 194ms/step - loss: 1.5065 - accuracy: 0.5682 - val_loss: 1.2201 - val_accuracy: 0.6903
Epoch 2/4
 39/128 [========>.....................] - ETA: 13s - loss: 1.0464 - accuracy: 0.7420Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0707 - accuracy: 0.7323Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 1.0242 - accuracy: 0.7510Unfroze 5 out of 12
128/128 [==============================] - 21s 164ms/step - loss: 1.0213 - accuracy: 0.7507 - val_loss: 0.9464 - val_accuracy: 0.7876
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.7894 - accuracy: 0.7973Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8528 - accuracy: 0.7761Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.8349 - accuracy: 0.7825Unfroze 8 out of 12
128/128 [==============================] - 21s 167ms/step - loss: 0.8402 - accuracy: 0.7792 - val_loss: 0.8383 - val_accuracy: 0.8142
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7445 - accuracy: 0.8054Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 7s - loss: 0.7518 - accuracy: 0.8093Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7523 - accuracy: 0.8093Unfroze 11 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.7577 - accuracy: 0.8057 - val_loss: 0.8070 - val_accuracy: 0.8363
Fold 9 -- Acc: 0.8363, Prec: 0.8216, Rec: 0.6637, F1: 0.6990
Starting fold 10/10
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Unfreezing will occur every 42 batches.
Epoch 1/4
 41/128 [========>.....................] - ETA: 14s - loss: 1.6613 - accuracy: 0.5762Unfroze 0 out of 12
 83/128 [==================>...........] - ETA: 7s - loss: 1.5070 - accuracy: 0.6107Unfroze 1 out of 12
125/128 [============================>.] - ETA: 0s - loss: 1.4240 - accuracy: 0.6320Unfroze 2 out of 12
128/128 [==============================] - 40s 189ms/step - loss: 1.4181 - accuracy: 0.6340 - val_loss: 1.3251 - val_accuracy: 0.7168
Epoch 2/4
 39/128 [========>.....................] - ETA: 14s - loss: 1.0251 - accuracy: 0.7548Unfroze 3 out of 12
 81/128 [=================>............] - ETA: 7s - loss: 1.0090 - accuracy: 0.7600Unfroze 4 out of 12
123/128 [===========================>..] - ETA: 0s - loss: 0.9887 - accuracy: 0.7642Unfroze 5 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.9852 - accuracy: 0.7640 - val_loss: 1.0747 - val_accuracy: 0.7434
Epoch 3/4
 37/128 [=======>......................] - ETA: 14s - loss: 0.8185 - accuracy: 0.7889Unfroze 6 out of 12
 79/128 [=================>............] - ETA: 7s - loss: 0.8219 - accuracy: 0.7896Unfroze 7 out of 12
121/128 [===========================>..] - ETA: 1s - loss: 0.7999 - accuracy: 0.7949Unfroze 8 out of 12
128/128 [==============================] - 21s 165ms/step - loss: 0.8030 - accuracy: 0.7954 - val_loss: 0.9975 - val_accuracy: 0.7566
Epoch 4/4
 35/128 [=======>......................] - ETA: 14s - loss: 0.7364 - accuracy: 0.8286Unfroze 9 out of 12
 77/128 [=================>............] - ETA: 8s - loss: 0.7437 - accuracy: 0.8182Unfroze 10 out of 12
119/128 [==========================>...] - ETA: 1s - loss: 0.7265 - accuracy: 0.8230Unfroze 11 out of 12
128/128 [==============================] - 21s 167ms/step - loss: 0.7240 - accuracy: 0.8238 - val_loss: 0.9555 - val_accuracy: 0.7566
Fold 10 -- Acc: 0.7566, Prec: 0.7072, Rec: 0.5742, F1: 0.5829

Cross-validation results:
Mean Accuracy: 0.7973 ± 0.0278
Mean Precision: 0.7488 ± 0.1030
Mean Recall: 0.6259 ± 0.0480
Mean F1: 0.6251 ± 0.0701

In [ ]:
fb.save_model("/content/drive/finbert")

OSError: [Errno 95] Operation not supported: '/content/drive/finbert'